# Fase 1 — Exploración y Diagnóstico del Dataset
**Dataset:** Olist Brazilian E-Commerce (Kaggle)  
**Objetivo:** Realizar una exploración inicial de los 9 CSVs, documentar problemas de calidad encontrados y generar un perfil completo de cada tabla antes de la limpieza.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Librerías cargadas correctamente')

Librerías cargadas correctamente


In [22]:
DATA_PATH = '../data/raw/'

df_customers = pd.read_csv(f'{DATA_PATH}olist_customers_dataset.csv')
df_geolocation = pd.read_csv(f'{DATA_PATH}olist_geolocation_dataset.csv')
df_order_items = pd.read_csv(f'{DATA_PATH}olist_order_items_dataset.csv')
df_payments = pd.read_csv(f'{DATA_PATH}olist_order_payments_dataset.csv')
df_reviews = pd.read_csv(f'{DATA_PATH}olist_order_reviews_dataset.csv', on_bad_lines='warn')
df_orders = pd.read_csv(f'{DATA_PATH}olist_orders_dataset.csv')
df_products = pd.read_csv(f'{DATA_PATH}olist_products_dataset.csv')
df_sellers = pd.read_csv(f'{DATA_PATH}olist_sellers_dataset.csv')
df_category_translation = pd.read_csv(f'{DATA_PATH}product_category_name_translation.csv')

datasets = {
    'customers': df_customers,
    'geolocation': df_geolocation,
    'order_items': df_order_items,
    'payments': df_payments,
    'reviews': df_reviews,
    'orders': df_orders,
    'products': df_products,
    'sellers': df_sellers,
    'category_translation': df_category_translation
}

print('Todos los datasets cargados correctamente')
print(f'Total de datasets: {len(datasets)}')

Todos los datasets cargados correctamente
Total de datasets: 9


## 3. Resumen General — Vista Panorámica

In [6]:
resumen = []
for nombre, df in datasets.items():
    resumen.append({
        'Dataset': nombre,
        'Filas': df.shape[0],
        'Columnas': df.shape[1],
        'Nulos totales': df.isnull().sum().sum(),
        '% Nulos': round(df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2),
        'Duplicados': df.duplicated().sum(),
        'Memoria (MB)': round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    })

df_resumen = pd.DataFrame(resumen)
df_resumen

,Dataset,Filas,Columnas,Nulos totales,% Nulos,Duplicados,Memoria (MB)
0,customers,99441,5,0,0.00,0,29.62
1,geolocation,1000163,5,0,0.00,261836,146.09
2,order_items,112650,7,0,0.00,0,44.87
3,payments,103886,5,0,0.00,0,17.81
4,reviews,99224,7,145903,21.01,0,42.18
5,orders,99441,8,4908,0.62,0,57.56
6,products,32951,9,2448,0.83,0,6.79
7,sellers,3095,4,0,0.00,0,0.66
8,category_translation,71,2,0,0.00,0,0.01


## 4. Función de Perfilado
Función reutilizable que genera el perfil completo de cualquier DataFrame.

In [81]:
def perfilar_dataset(df, nombre):
    print('=' * 70)
    print(f'PERFIL: {nombre.upper()}')
    print('=' * 70)
    
    print(f'\nDimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas')
    print(f'Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
    
    print(f'\nTipos de datos:')
    for col in df.columns:
        print(f'   {col}: {df[col].dtype}')
    
    nulos = df.isnull().sum()
    if nulos.sum() > 0:
        print(f'\nValores nulos:')
        for col in nulos[nulos > 0].index:
            pct = nulos[col] / len(df) * 100
            print(f'   {col}: {nulos[col]:,} ({pct:.1f}%)')
    else:
        print(f'\nSin valores nulos')
    
    dupes = df.duplicated().sum()
    print(f'\nFilas duplicadas completas: {dupes:,}')
    
    print(f'\nValores unicos por columna:')
    for col in df.columns:
        print(f'   {col}: {df[col].nunique():,}')
    
    id_cols = [col for col in df.columns if col.endswith('_id')]
    if id_cols:
        print(f'\nValidacion de IDs:')
        for col in id_cols:
            valores = df[col].dropna().astype(str)
            longitudes = valores.str.len().value_counts().sort_index()
            con_espacios = valores.str.contains(' ', na=False).sum()
            con_comillas = valores.str.contains('"', na=False).sum()
            todas_hex = valores.str.match(r'^[a-f0-9]+$').all()
            
            problemas = []
            if len(longitudes) > 1:
                problemas.append(f'longitudes mixtas: {dict(longitudes)}')
            if con_espacios > 0:
                problemas.append(f'{con_espacios} con espacios')
            if con_comillas > 0:
                problemas.append(f'{con_comillas} con comillas')
            if not todas_hex:
                problemas.append('contiene caracteres no hexadecimales')
            
            if problemas:
                print(f'      {col}: {", ".join(problemas)}')
            else:
                longitud = longitudes.index[0]
                print(f'     {col}: OK ({longitud} chars, hexadecimal)')
    
    print(f'\nPrimeras 3 filas:')
    display(df.head(3))
    
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        print(f'\nEstadisticas descriptivas (numericas):')
        display(df[num_cols].describe())
    
    print()

## 5. Exploración Detallada por Dataset
### 5.1 Customers (Clientes)

In [82]:
perfilar_dataset(df_customers, 'customers')

PERFIL: CUSTOMERS

Dimensiones: 99,441 filas x 5 columnas
Memoria: 29.62 MB

Tipos de datos:
   customer_id: str
   customer_unique_id: str
   customer_zip_code_prefix: int64
   customer_city: str
   customer_state: str

Sin valores nulos

Filas duplicadas completas: 0

Valores unicos por columna:
   customer_id: 99,441
   customer_unique_id: 96,096
   customer_zip_code_prefix: 14,994
   customer_city: 4,119
   customer_state: 27

Validacion de IDs:
     customer_id: OK (32 chars, hexadecimal)
     customer_unique_id: OK (32 chars, hexadecimal)

Primeras 3 filas:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



Estadisticas descriptivas (numericas):


,customer_zip_code_prefix
count,99441.00
mean,35137.47
std,29797.94
min,1003.00
25%,11347.00
50%,24416.00
75%,58900.00
max,99990.00


In [9]:
print(f'customer_id unicos: {df_customers["customer_id"].nunique():,}')
print(f'customer_unique_id unicos: {df_customers["customer_unique_id"].nunique():,}')
print(f'Total filas: {len(df_customers):,}')
print(f'\nHay {len(df_customers) - df_customers["customer_unique_id"].nunique():,} clientes que hicieron mas de una compra')

customer_id unicos: 99,441
customer_unique_id unicos: 96,096
Total filas: 99,441

Hay 3,345 clientes que hicieron mas de una compra


In [63]:
print('Top 10 estados por cantidad de clientes:')
top_estados = df_customers['customer_state'].value_counts().head(10)
total = len(df_customers)
for estado, count in top_estados.items():
    barra = '█' * int(count / total * 50)
    print(f'   {estado}: {count:,} ({count/total*100:.1f}%) {barra}')

Top 10 estados por cantidad de clientes:
   SP: 41,746 (42.0%) ████████████████████
   RJ: 12,852 (12.9%) ██████
   MG: 11,635 (11.7%) █████
   RS: 5,466 (5.5%) ██
   PR: 5,045 (5.1%) ██
   SC: 3,637 (3.7%) █
   BA: 3,380 (3.4%) █
   DF: 2,140 (2.2%) █
   ES: 2,033 (2.0%) █
   GO: 2,020 (2.0%) █


In [11]:
print(f'\nProblemas de ciudades detectadas:')
print(f'   1. Todas las ciudades estan en minusculas (no hay capitalizacion)')
print(f'   2. No se usan diacríticos/tíldes (ej: "sao paulo" en lugar de "são paulo")')
print(f'     Decisión de diseño: mantener minusculas y sin diacriticos para normalizar')
print(f'     y facilitar JOINs entre tablas')


Problemas de ciudades detectadas:
   1. Todas las ciudades estan en minusculas (no hay capitalizacion)
   2. No se usan diacríticos/tíldes (ej: "sao paulo" en lugar de "são paulo")
     Decisión de diseño: mantener minusculas y sin diacriticos para normalizar
     y facilitar JOINs entre tablas


### 5.2 Geolocation (Geolocalización)

In [83]:
perfilar_dataset(df_geolocation, 'geolocation')

PERFIL: GEOLOCATION

Dimensiones: 1,000,163 filas x 5 columnas
Memoria: 145.20 MB

Tipos de datos:
   geolocation_zip_code_prefix: int64
   geolocation_lat: float64
   geolocation_lng: float64
   geolocation_city: str
   geolocation_state: str

Sin valores nulos

Filas duplicadas completas: 261,836

Valores unicos por columna:
   geolocation_zip_code_prefix: 19,015
   geolocation_lat: 716,685
   geolocation_lng: 717,097
   geolocation_city: 8,011
   geolocation_state: 27

Primeras 3 filas:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP



Estadisticas descriptivas (numericas):


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
count,1000163.00,1000163.00,1000163.00
mean,36574.17,-21.18,-46.39
std,30549.34,5.72,4.27
min,1001.00,-36.61,-101.47
25%,11075.00,-23.60,-48.57
50%,26530.00,-22.92,-46.64
75%,63504.00,-19.98,-43.77
max,99990.00,45.07,121.11


### Brasil deberia estar entre:
### Latitud: -33.75 a 5.27
### Longitud: -73.99 a -34.79

In [13]:
lat_min, lat_max = -33.75, 5.27
lng_min, lng_max = -73.99, -34.79

fuera_lat = df_geolocation[
    (df_geolocation['geolocation_lat'] < lat_min) | 
    (df_geolocation['geolocation_lat'] > lat_max)
]
fuera_lng = df_geolocation[
    (df_geolocation['geolocation_lng'] < lng_min) | 
    (df_geolocation['geolocation_lng'] > lng_max)
]

print(f'  Coordenadas fuera del rango de Brasil:')
print(f'   Latitud fuera de rango: {len(fuera_lat):,}')
print(f'   Longitud fuera de rango: {len(fuera_lng):,}')

if len(fuera_lat) > 0:
    print(f'\nEjemplos de latitudes fuera de rango:')
    display(fuera_lat.head())
if len(fuera_lng) > 0:
    print(f'\nEjemplos de longitudes fuera de rango:')
    display(fuera_lng.head())

  Coordenadas fuera del rango de Brasil:
   Latitud fuera de rango: 31
   Longitud fuera de rango: 37

Ejemplos de latitudes fuera de rango:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.01,-15.54,bom retiro da esperanca,SP
513631,28165,41.61,-8.41,vila nova de campos,RJ
513643,28155,-34.59,-58.73,santa maria,RJ
513754,28155,42.44,13.82,santa maria,RJ
514429,28333,38.38,-6.33,raposo,RJ



Ejemplos de longitudes fuera de rango:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.01,-15.54,bom retiro da esperanca,SP
513631,28165,41.61,-8.41,vila nova de campos,RJ
513754,28155,42.44,13.82,santa maria,RJ
514429,28333,38.38,-6.33,raposo,RJ
516682,28595,43.68,-7.41,portela,RJ


In [14]:
print(f'Zip codes unicos: {df_geolocation["geolocation_zip_code_prefix"].nunique():,}')
print(f'Total filas: {len(df_geolocation):,}')
print(f'Filas duplicadas completas: {df_geolocation.duplicated().sum():,}')
print(f'Promedio de registros por zip code: {len(df_geolocation) / df_geolocation["geolocation_zip_code_prefix"].nunique():.1f}')

Zip codes unicos: 19,015
Total filas: 1,000,163
Filas duplicadas completas: 261,836
Promedio de registros por zip code: 52.6


In [15]:
ciudades = df_geolocation['geolocation_city'].unique()
con_diacriticos = [c for c in ciudades if any(ch in str(c) for ch in 'áéíóúãõâêôçñü£')]
sin_match = [c for c in ciudades if '£' in str(c)]

print(f'Total ciudades unicas: {len(ciudades):,}')
print(f'Ciudades con diacriticos/acentos: {len(con_diacriticos)}')
print(f'Ciudades con caracter £ (encoding corrupto): {len(sin_match)}')

if len(sin_match) > 0:
    print(f'\n  Ciudades con encoding corrupto (£):')
    for c in sin_match:
        print(f'     "{c}"')

# Ejemplo concreto del problema de duplicados
print(f'\nEjemplo del problema — variantes de la misma ciudad:')
print(f'   "sao paulo" vs "são paulo" vs "sãopaulo" vs "sa£o paulo"')
print(f'     Son la misma ciudad con 4 escrituras diferentes')

Total ciudades unicas: 8,011
Ciudades con diacriticos/acentos: 2082
Ciudades con caracter £ (encoding corrupto): 1

  Ciudades con encoding corrupto (£):
     "sa£o paulo"

Ejemplo del problema — variantes de la misma ciudad:
   "sao paulo" vs "são paulo" vs "sãopaulo" vs "sa£o paulo"
     Son la misma ciudad con 4 escrituras diferentes


In [16]:
ciudades_con_coma = df_geolocation[
    df_geolocation['geolocation_city'].str.contains(',', na=False)
]
print(f'  Ciudades con formato incorrecto (contienen comas): {len(ciudades_con_coma):,}')
if len(ciudades_con_coma) > 0:
    for ciudad in ciudades_con_coma['geolocation_city'].unique():
        print(f'   → "{ciudad}"')

  Ciudades con formato incorrecto (contienen comas): 2
   → "rio de janeiro, rio de janeiro, brasil"
   → "campo alegre de lourdes, bahia, brasil"


In [17]:
print(f'   Problemas de estandarizacion de texto en Geolocation:')
print(f'   Total ciudades unicas: {len(df_geolocation["geolocation_city"].unique()):,}')
print(f'   Ciudades con diacriticos/acentos: 2,082 (26%)')
print(f'   Ciudades con encoding corrupto (£): 1')
print(f'')
print(f'   Tipos de inconsistencia detectados:')
print(f'   1. Con acento vs sin acento: "são paulo" vs "sao paulo"')
print(f'   2. Encoding corrupto: "sa£o paulo"')
print(f'   3. Espacios faltantes: "sãopaulo"')
print(f'     Esto genera ciudades duplicadas bajo nombres diferentes')
print(f'     Decisión Fase 2: normalizar todo a minusculas sin diacriticos')
print(f'   4. Precisión inconsistente en coordenadas (entre 2 y 15 decimales)')
print(f'     Decisión Fase 2: estandarizar a 6 decimales (precision de ~10cm)')
print(f'     Las coordenadas con menos de 6 decimales se mantienen tal cual')
print(f'   5. 2 ciudades con formato incorrecto: incluyen estado y pais en el nombre')
print(f'        "rio de janeiro, rio de janeiro, brasil"')
print(f'        "campo alegre de lourdes, bahia, brasil"')
print(f'     Decisión Fase 2: extraer solo el nombre de la ciudad antes de la primera coma')

   Problemas de estandarizacion de texto en Geolocation:
   Total ciudades unicas: 8,011
   Ciudades con diacriticos/acentos: 2,082 (26%)
   Ciudades con encoding corrupto (£): 1

   Tipos de inconsistencia detectados:
   1. Con acento vs sin acento: "são paulo" vs "sao paulo"
   2. Encoding corrupto: "sa£o paulo"
   3. Espacios faltantes: "sãopaulo"
     Esto genera ciudades duplicadas bajo nombres diferentes
     Decisión Fase 2: normalizar todo a minusculas sin diacriticos
   4. Precisión inconsistente en coordenadas (entre 2 y 15 decimales)
     Decisión Fase 2: estandarizar a 6 decimales (precision de ~10cm)
     Las coordenadas con menos de 6 decimales se mantienen tal cual
   5. 2 ciudades con formato incorrecto: incluyen estado y pais en el nombre
        "rio de janeiro, rio de janeiro, brasil"
        "campo alegre de lourdes, bahia, brasil"
     Decisión Fase 2: extraer solo el nombre de la ciudad antes de la primera coma


In [18]:
print(f'   Se verifico que las coordenadas usan punto decimal correctamente (no hay comas como separador)')

   Se verifico que las coordenadas usan punto decimal correctamente (no hay comas como separador)


### 5.3 Order Items (Ítems de Órdenes)

In [84]:
perfilar_dataset(df_order_items, 'order_items')

PERFIL: ORDER_ITEMS

Dimensiones: 112,650 filas x 7 columnas
Memoria: 39.11 MB

Tipos de datos:
   order_id: str
   order_item_id: int64
   product_id: str
   seller_id: str
   shipping_limit_date: str
   price: float64
   freight_value: float64

Valores nulos:
   freight_value: 1 (0.0%)

Filas duplicadas completas: 0

Valores unicos por columna:
   order_id: 98,666
   order_item_id: 21
   product_id: 32,951
   seller_id: 3,095
   shipping_limit_date: 54,615
   price: 5,968
   freight_value: 6,998

Validacion de IDs:
     order_id: OK (32 chars, hexadecimal)
      order_item_id: longitudes mixtas: {1: np.int64(112559), 2: np.int64(91)}
     product_id: OK (32 chars, hexadecimal)
     seller_id: OK (32 chars, hexadecimal)

Primeras 3 filas:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,19/09/2017 09:45,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,03/05/2017 11:05,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,18/01/2018 14:48,199.00,17.87



Estadisticas descriptivas (numericas):


,order_item_id,price,freight_value
count,112650.00,112650.00,112649.00
mean,1.20,120.65,19.99
std,0.71,183.63,15.80
min,1.00,0.85,0.00
25%,1.00,39.90,13.08
50%,1.00,74.99,16.26
75%,1.00,134.90,21.15
max,21.00,6735.00,409.68


In [35]:
print(f'Tipo de price: {df_order_items["price"].dtype}')
print(f'Tipo de freight_value: {df_order_items["freight_value"].dtype}')

df_order_items['freight_value'] = pd.to_numeric(df_order_items['freight_value'], errors='coerce')

no_convertidos = df_order_items['freight_value'].isna().sum()
print(f'\nValores de freight_value que no se pudieron convertir a numero: {no_convertidos}')

# Ahora si la validacion
print(f'\nValidacion de valores monetarios:')
print(f'   price < 0: {(df_order_items["price"] < 0).sum()}')
print(f'   price = 0: {(df_order_items["price"] == 0).sum()}')
print(f'   freight_value < 0: {(df_order_items["freight_value"] < 0).sum()}')
print(f'   freight_value = 0: {(df_order_items["freight_value"] == 0).sum()}')

Tipo de price: float64
Tipo de freight_value: str

Valores de freight_value que no se pudieron convertir a numero: 1

Validacion de valores monetarios:
   price < 0: 0
   price = 0: 0
   freight_value < 0: 0
   freight_value = 0: 383


In [37]:
original = pd.read_csv('../data/raw/olist_order_items_dataset.csv', dtype=str)
problematico = original[pd.to_numeric(original['freight_value'], errors='coerce').isna()]
print(f'Registro con freight_value corrupto:')
display(problematico)
print(f'\nValor exacto de freight_value: "{problematico["freight_value"].values[0]}"')

Registro con freight_value corrupto:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,1,489ae2aa008f021502940f251d4cce7f,e3b4998c7a498169dc7bce44e6bb6277,16/02/2017 20:37,6735,



Valor exacto de freight_value: " "


In [38]:
print(f'   1 registro con freight_value vacio (order_id: 12eb902a67711a1cb742b3cdaa65ae14)')
print(f'   → Es el mismo registro con el precio mas alto del dataset (R$6,735)')
print(f'   → Decision Fase 2: investigar la orden completa y decidir si imputar o eliminar')

   1 registro con freight_value vacio (order_id: 12eb902a67711a1cb742b3cdaa65ae14)
   → Es el mismo registro con el precio mas alto del dataset (R$6,735)
   → Decision Fase 2: investigar la orden completa y decidir si imputar o eliminar


In [39]:
items_por_orden = df_order_items.groupby('order_id')['order_item_id'].max()
print(f'\nOrdenes con mas de 5 items: {(items_por_orden > 5).sum():,}')


Ordenes con mas de 5 items: 256


In [26]:
print(f'Tipo actual de shipping_limit_date: {df_order_items["shipping_limit_date"].dtype}')
print(f'\nEjemplos:')
for fecha in df_order_items['shipping_limit_date'].head(5):
    print(f'   → {fecha}')

Tipo actual de shipping_limit_date: str

Ejemplos:
   → 19/09/2017 09:45
   → 03/05/2017 11:05
   → 18/01/2018 14:48
   → 15/08/2018 10:10
   → 13/02/2017 13:57


In [27]:
print(f'   Hallazgos en Order Items:')
print(f'   1. shipping_limit_date esta como texto, debe convertirse a datetime')
print(f'   2. {(df_order_items["freight_value"] == 0).sum()} items con flete = 0 (envio gratuito o error)')
print(f'   3. Precio maximo: R${df_order_items["price"].max():,.2f} (USD $1279.66) — verificar si es outlier o legitimo')
print(f'   4. La mayoria de ordenes tienen 1 solo item (mediana = 1, promedio = 1.14)')
print(f'     Sin valores nulos')
print(f'     Sin filas duplicadas')
print(f'   5. Sin precios negativos ni en cero')
print(f'   6. 383 items con flete = 0 (0.3%) — probablemente envio gratuito o retiro en tienda')

   Hallazgos en Order Items:
   1. shipping_limit_date esta como texto, debe convertirse a datetime
   2. 0 items con flete = 0 (envio gratuito o error)
   3. Precio maximo: R$6,735.00 (USD $1279.66) — verificar si es outlier o legitimo
   4. La mayoria de ordenes tienen 1 solo item (mediana = 1, promedio = 1.14)
     Sin valores nulos
     Sin filas duplicadas
   5. Sin precios negativos ni en cero
   6. 383 items con flete = 0 (0.3%) — probablemente envio gratuito o retiro en tienda


In [85]:
fechas_convertidas = pd.to_datetime(df_order_items['shipping_limit_date'], errors='coerce')
fechas_invalidas = fechas_convertidas.isna().sum()

print(f'Formato raw del CSV: DD/MM/YYYY HH:MM (24h)')
print(f'Formato en Python: YYYY-MM-DD HH:MM:SS')
print(f'Fechas invalidas al convertir: {fechas_invalidas}')

print(f'\nRango de fechas:')
print(f'   Mas antigua: {fechas_convertidas.min()}')
print(f'   Mas reciente: {fechas_convertidas.max()}')

print(f'\nDistribucion por mes (para verificar que no se confundio dia/mes):')
meses = fechas_convertidas.dt.month.value_counts().sort_index()
for mes, count in meses.items():
    print(f'   Mes {mes:2d}: {count:,}')

Formato raw del CSV: DD/MM/YYYY HH:MM (24h)
Formato en Python: YYYY-MM-DD HH:MM:SS
Fechas invalidas al convertir: 0

Rango de fechas:
   Mas antigua: 2016-09-19 00:15:00
   Mas reciente: 2020-04-09 22:35:00

Distribucion por mes (para verificar que no se confundio dia/mes):
   Mes  1: 8,173
   Mes  2: 9,243
   Mes  3: 11,510
   Mes  4: 10,003
   Mes  5: 12,915
   Mes  6: 10,698
   Mes  7: 10,788
   Mes  8: 13,857
   Mes  9: 4,827
   Mes 10: 5,554
   Mes 11: 7,355
   Mes 12: 7,727


In [64]:
print(f'Distribucion por año:')
total_fechas = len(fechas_convertidas)
anios = fechas_convertidas.dt.year.value_counts().sort_index()
for anio, count in anios.items():
    barra = '█' * int(count / total_fechas * 50)
    print(f'   {anio}: {count:,} ({count/total_fechas*100:.1f}%) {barra}')

Distribucion por año:
   2016: 370 (0.3%) 
   2017: 49,765 (44.2%) ██████████████████████
   2018: 62,511 (55.5%) ███████████████████████████
   2020: 4 (0.0%) 


In [34]:
registros_2020 = df_order_items[fechas_convertidas.dt.year == 2020]
display(registros_2020)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
8643,13bdf405f961a6deec817d817f5c6624,1,96ea060e41bdecc64e2de00b97068975,7a241947449cc45dbfda4f9d0798d9d0,05/02/2020 03:30,69.99,14.66
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,1,282b126b2354516c5f400154398f616d,7a241947449cc45dbfda4f9d0798d9d0,03/02/2020 20:23,75.99,14.7
85729,c2bb89b5c1dd978d507284be78a04cb2,1,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,09/04/2020 22:35,99.99,61.44
85730,c2bb89b5c1dd978d507284be78a04cb2,2,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,09/04/2020 22:35,99.99,61.44


In [32]:
print(f'   Hallazgos en fechas de Order Items:')
print(f'   Formato raw: DD/MM/YYYY HH:MM (24h)')
print(f'   Rango: 2016-09-19 a 2020-04-09')
print(f'   Fechas invalidas: 0')
print(f'     Pandas interpreto correctamente dia/mes (distribucion por mes es coherente)')
print(f'      4 registros con shipping_limit_date en 2020 (el dataset es 2016-2018)')
print(f'     DecisiÓn Fase 2: investigar si son errores y corregir o eliminar')

   Hallazgos en fechas de Order Items:
   Formato raw: DD/MM/YYYY HH:MM (24h)
   Rango: 2016-09-19 a 2020-04-09
   Fechas invalidas: 0
     Pandas interpreto correctamente dia/mes (distribucion por mes es coherente)
      4 registros con shipping_limit_date en 2020 (el dataset es 2016-2018)
     DecisiÓn Fase 2: investigar si son errores y corregir o eliminar


In [40]:
flete_mayor = df_order_items[df_order_items['freight_value'] > df_order_items['price']]
print(f'Items donde el flete es mayor al precio del producto: {len(flete_mayor):,} ({len(flete_mayor)/len(df_order_items)*100:.1f}%)')
print(f'\nEjemplos:')
display(flete_mayor[['order_id', 'price', 'freight_value']].head(10))

Items donde el flete es mayor al precio del producto: 4,124 (3.7%)

Ejemplos:


,order_id,price,freight_value
58,0025081dcf9330f9a5052ae82c6ce396,14.95,18.23
80,002f98c0f7efd42638ed6100ca699b42,8.99,32.57
110,003edccf16bc5ec447f592913b3df2b4,14.00,50.85
125,00482f2670787292280e0a8153d82467,7.60,10.96
156,00602f25bffa1dcfb71e202fbf9824fb,39.90,54.02
193,0080eebb288dba1857ccf048dfe6bdfe,7.00,14.10
204,008a1b3db2a8bf63418c2cf7c7f494b1,14.89,18.23
209,008d9bf350ff02ed444b3452cf3f57e0,9.99,15.23
210,008d9bf350ff02ed444b3452cf3f57e0,9.99,15.23
211,008dc31c0ae7121f2791953ae5f532c3,28.50,34.15


In [43]:
print(f'   {len(flete_mayor):,} items donde el flete supera el precio del producto (3.7%)')
print(f'   → No es un error: es comun en e-commerce brasileño por las distancias')
print(f'   → Relevante para análisis de negocio en el dashboard (Fase 4)')

   4,124 items donde el flete supera el precio del producto (3.7%)
   → No es un error: es comun en e-commerce brasileño por las distancias
   → Relevante para análisis de negocio en el dashboard (Fase 4)


### 5.4 Payments (Pagos)

In [86]:
perfilar_dataset(df_payments, 'payments')

PERFIL: PAYMENTS

Dimensiones: 103,886 filas x 5 columnas
Memoria: 17.81 MB

Tipos de datos:
   order_id: str
   payment_sequential: int64
   payment_type: str
   payment_installments: int64
   payment_value: float64

Sin valores nulos

Filas duplicadas completas: 0

Valores unicos por columna:
   order_id: 99,440
   payment_sequential: 29
   payment_type: 5
   payment_installments: 24
   payment_value: 29,077

Validacion de IDs:
     order_id: OK (32 chars, hexadecimal)

Primeras 3 filas:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71



Estadisticas descriptivas (numericas):


,payment_sequential,payment_installments,payment_value
count,103886.00,103886.00,103886.00
mean,1.09,2.85,154.10
std,0.71,2.69,217.49
min,1.00,0.00,0.00
25%,1.00,1.00,56.79
50%,1.00,1.00,100.00
75%,1.00,4.00,171.84
max,29.00,24.00,13664.08


In [65]:
print('Distribucion de tipos de pago:')
total_pagos = len(df_payments)
for tipo, count in df_payments['payment_type'].value_counts().items():
    barra = '█' * int(count / total_pagos * 50)
    print(f'   {tipo}: {count:,} ({count/total_pagos*100:.1f}%) {barra}')

Distribucion de tipos de pago:
   credit_card: 76,795 (73.9%) ████████████████████████████████████
   boleto: 19,784 (19.0%) █████████
   voucher: 5,775 (5.6%) ██
   debit_card: 1,529 (1.5%) 
   not_defined: 3 (0.0%) 


In [46]:
pagos_por_orden = df_payments.groupby('order_id')['payment_sequential'].max()
multiples = (pagos_por_orden > 1).sum()
print(f'Ordenes con multiples metodos de pago: {multiples:,} ({multiples/len(pagos_por_orden)*100:.1f}%)')
print(f'\nDistribucion de cantidad de pagos por orden:')
print(pagos_por_orden.value_counts().sort_index())

Ordenes con multiples metodos de pago: 3,039 (3.1%)

Distribucion de cantidad de pagos por orden:
payment_sequential
1     96401
2      2458
3       303
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64


In [49]:
pagos_cero = df_payments[df_payments['payment_value'] == 0]
print(f'Validacion de payment_value:')
print(f'   Valores < 0: {(df_payments["payment_value"] < 0).sum()}')
print(f'   Valores = 0: {len(pagos_cero)} ({len(pagos_cero)/len(df_payments)*100:.2f}%)')
print(f'   Installments = 0: {(df_payments["payment_installments"] == 0).sum()}')

print(f'\n   Pagos con valor 0: {len(pagos_cero)}')
print(f'   Tipos de pago en valor 0:')
for tipo, count in pagos_cero['payment_type'].value_counts().items():
    print(f'      {tipo}: {count}')

display(pagos_cero)

Validacion de payment_value:
   Valores < 0: 0
   Valores = 0: 9 (0.01%)
   Installments = 0: 2

   Pagos con valor 0: 9
   Tipos de pago en valor 0:
      voucher: 6
      not_defined: 3


,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.00


In [50]:
print('Promedio de installments por tipo de pago:')
for tipo in df_payments['payment_type'].unique():
    subset = df_payments[df_payments['payment_type'] == tipo]
    print(f'\n   {tipo}:')
    print(f'      Registros: {len(subset):,}')
    print(f'      Installments = 1: {(subset["payment_installments"] == 1).sum():,}')
    print(f'      Installments > 1: {(subset["payment_installments"] > 1).sum():,}')
    print(f'      Installments = 0: {(subset["payment_installments"] == 0).sum():,}')
    print(f'      Max installments: {subset["payment_installments"].max()}')

Promedio de installments por tipo de pago:

   credit_card:
      Registros: 76,795
      Installments = 1: 25,455
      Installments > 1: 51,338
      Installments = 0: 2
      Max installments: 24

   boleto:
      Registros: 19,784
      Installments = 1: 19,784
      Installments > 1: 0
      Installments = 0: 0
      Max installments: 1

   voucher:
      Registros: 5,775
      Installments = 1: 5,775
      Installments > 1: 0
      Installments = 0: 0
      Max installments: 1

   debit_card:
      Registros: 1,529
      Installments = 1: 1,529
      Installments > 1: 0
      Installments = 0: 0
      Max installments: 1

   not_defined:
      Registros: 3
      Installments = 1: 3
      Installments > 1: 0
      Installments = 0: 0
      Max installments: 1


In [55]:
not_defined = df_payments[df_payments['payment_type'] == 'not_defined']
ordenes_nd = not_defined['order_id'].tolist()

print('Todos los pagos de las ordenes con not_defined:')
display(df_payments[df_payments['order_id'].isin(ordenes_nd)])

print(f'\nEstatus de esas ordenes:')
for oid in ordenes_nd:
    status = df_orders[df_orders['order_id'] == oid]['order_status'].values
    print(f'   {oid}: {status}')

Todos los pagos de las ordenes con not_defined:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00



Estatus de esas ordenes:
   4637ca194b6387e2d538dc89b124b0ee: <StringArray>
['canceled']
Length: 1, dtype: str
   00b1cb0320190ca0daa2c88b35206009: <StringArray>
['canceled']
Length: 1, dtype: str
   c8c528189310eaa44a745b8d9d26908b: <StringArray>
['canceled']
Length: 1, dtype: str


In [56]:
print(f'   Resumen de hallazgos en Payments:')
print(f'   1. 3 pagos con tipo "not_defined" — tipo de pago no identificado')
print(f'   2. 9 pagos con valor 0 (6 vouchers + 3 not_defined)')
print(f'   3. 2 tarjetas de credito con installments = 0 (deberia ser minimo 1)')
print(f'     Solo credit_card usa cuotas (installments > 1), los demas siempre son 1')
print(f'     Boleto, voucher, debit_card y not_defined siempre tienen installments = 1')
print(f'     Sin valores negativos en montos')
print(f'     Sin nulos ni duplicados')
print(f'      73.9% de pagos son con tarjeta de credito')
print(f'      3,039 ordenes (3.1%) usan multiples metodos de pago')
print(f'   → Decision Fase 2: eliminar los 3 registros not_defined (ordenes canceladas sin valor)')

   Resumen de hallazgos en Payments:
   1. 3 pagos con tipo "not_defined" — tipo de pago no identificado
   2. 9 pagos con valor 0 (6 vouchers + 3 not_defined)
   3. 2 tarjetas de credito con installments = 0 (deberia ser minimo 1)
     Solo credit_card usa cuotas (installments > 1), los demas siempre son 1
     Boleto, voucher, debit_card y not_defined siempre tienen installments = 1
     Sin valores negativos en montos
     Sin nulos ni duplicados
      73.9% de pagos son con tarjeta de credito
      3,039 ordenes (3.1%) usan multiples metodos de pago
   → Decision Fase 2: eliminar los 3 registros not_defined (ordenes canceladas sin valor)


### 5.5 Reviews (Reseñas)

In [87]:
perfilar_dataset(df_reviews, 'reviews')

PERFIL: REVIEWS

Dimensiones: 99,224 filas x 7 columnas
Memoria: 29.88 MB

Tipos de datos:
   review_id: str
   order_id: str
   review_score: int64
   review_comment_title: str
   review_comment_message: str
   review_creation_date: datetime64[us]
   review_answer_timestamp: datetime64[us]

Valores nulos:
   review_comment_title: 87,656 (88.3%)
   review_comment_message: 58,247 (58.7%)

Filas duplicadas completas: 0

Valores unicos por columna:
   review_id: 98,410
   order_id: 98,673
   review_score: 5
   review_comment_title: 4,516
   review_comment_message: 36,155
   review_creation_date: 636
   review_answer_timestamp: 88,867

Validacion de IDs:
     review_id: OK (32 chars, hexadecimal)
     order_id: OK (32 chars, hexadecimal)

Primeras 3 filas:


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:00
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:00
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:00



Estadisticas descriptivas (numericas):


,review_score
count,99224.00
mean,4.09
std,1.35
min,1.00
25%,4.00
50%,5.00
75%,5.00
max,5.00


In [88]:
print(f'Reviews cargadas: {len(df_reviews):,}')
print(f'order_id unicos en reviews: {df_reviews["order_id"].nunique():,}')
print(f'order_id unicos en orders: {df_orders["order_id"].nunique():,}')

ordenes_con_review = set(df_reviews['order_id'].unique())
ordenes_total = set(df_orders['order_id'].unique())
sin_review = ordenes_total - ordenes_con_review

print(f'\nOrdenes sin review: {len(sin_review):,}')
print(f'→ Pueden ser ordenes no entregadas o reviews perdidas por filas corruptas')

Reviews cargadas: 99,224
order_id unicos en reviews: 98,673
order_id unicos en orders: 99,441

Ordenes sin review: 768
→ Pueden ser ordenes no entregadas o reviews perdidas por filas corruptas


In [69]:
print('Ejemplos de ordenes delivered sin review:')
delivered_sin_review = ordenes_sin_review[ordenes_sin_review['order_status'] == 'delivered']
display(delivered_sin_review.head(10))

Ejemplos de ordenes delivered sin review:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
16,403b97836b0c04a622354cf531062e5f,738b086814c6fcc74b8cc583f8516ee3,delivered,02/01/2018 19:00,02/01/2018 19:09,03/01/2018 18:19,20/01/2018 01:38,06/02/2018 00:00
311,4906eeadde5f70b308c20c4a8f20be02,4e7656e34357b93f14b40c6400ca3f6e,delivered,08/12/2017 04:45,12/12/2017 03:50,12/12/2017 17:43,09/01/2018 18:04,03/01/2018 00:00
382,b7a4a9ecb1cd3ef6a3e36a48e200e3be,c3d8fc500d86b1c961ee144395c13a57,delivered,19/05/2017 18:13,20/05/2017 11:35,30/05/2017 12:43,08/06/2017 07:53,16/06/2017 00:00
390,59b32faedc12322c672e95ec3716d614,5baa82a2c45fa3220cb57d9881db3211,delivered,27/06/2018 11:10,28/06/2018 02:15,28/06/2018 14:57,06/07/2018 16:37,26/07/2018 00:00
409,c2215076050fa358934105b15c34cf3b,19e7a88b34ef70d108e660c6eb33e82a,delivered,16/07/2017 10:04,16/07/2017 10:15,20/07/2017 17:42,26/07/2017 20:32,04/08/2017 00:00
536,eafef0e6c44f121531569a69a318c3b3,084dab2db2bf5d42677b135e633a15ba,delivered,29/11/2017 13:03,29/11/2017 13:12,04/12/2017 19:18,09/01/2018 16:32,27/12/2017 00:00
833,95d9d2979d40161be87292ff88563cba,e4721b61e8d2b4b85d606809f6baa292,delivered,07/06/2018 23:42,07/06/2018 23:53,08/06/2018 14:15,16/06/2018 16:44,26/06/2018 00:00
835,a9b151f0c0471d9b2534fa73c9c0e123,2238c417bc745d8fa7d7389d8b22154b,delivered,09/03/2017 16:18,09/03/2017 16:18,10/03/2017 10:35,16/03/2017 12:45,06/04/2017 00:00
1042,5e8d0f2f1e06e715aee3eefe4c175e52,743dd95aefb65359afe92a764748251a,delivered,28/11/2017 22:50,28/11/2017 23:10,30/11/2017 21:12,26/12/2017 17:38,19/12/2017 00:00
1128,8187896518689722d7c05bfbaeed0933,c499f24d5aca03c90fa6ee05d32d0988,delivered,29/11/2017 14:11,01/12/2017 11:31,01/12/2017 18:11,26/12/2017 19:22,22/12/2017 00:00


In [67]:
ordenes_sin_review = df_orders[~df_orders['order_id'].isin(df_reviews['order_id'])]
print(f'Status de las {len(ordenes_sin_review)} ordenes sin review:')
for status, count in ordenes_sin_review['order_status'].value_counts().items():
    print(f'   {status}: {count}')

Status de las 768 ordenes sin review:
   delivered: 646
   shipped: 75
   canceled: 20
   unavailable: 14
   processing: 6
   invoiced: 5
   created: 2


In [59]:
print('Distribucion de review_score:')
total_reviews = len(df_reviews)
for score, count in df_reviews['review_score'].value_counts().sort_index().items():
    barra = '█' * int(count / total_reviews * 50)
    print(f'   Score {score}: {count:,} ({count/total_reviews*100:.1f}%) {barra}')
print(f'\nPromedio: {df_reviews["review_score"].mean():.2f}')

Distribucion de review_score:
   Score 1: 11,424 (11.5%) █████
   Score 2: 3,151 (3.2%) █
   Score 3: 8,179 (8.2%) ████
   Score 4: 19,142 (19.3%) █████████
   Score 5: 57,328 (57.8%) ████████████████████████████

Promedio: 4.09


In [60]:
con_titulo = df_reviews['review_comment_title'].notna().sum()
con_mensaje = df_reviews['review_comment_message'].notna().sum()
total = len(df_reviews)

print(f'Reviews con titulo: {con_titulo:,} ({con_titulo/total*100:.1f}%)')
print(f'Reviews con mensaje: {con_mensaje:,} ({con_mensaje/total*100:.1f}%)')
print(f'Reviews sin ningun comentario: {total - con_mensaje:,} ({(total-con_mensaje)/total*100:.1f}%)')

Reviews con titulo: 11,568 (11.7%)
Reviews con mensaje: 40,977 (41.3%)
Reviews sin ningun comentario: 58,247 (58.7%)


In [61]:
dupes_review = df_reviews['review_id'].duplicated().sum()
print(f'review_id duplicados: {dupes_review:,}')
if dupes_review > 0:
    print(f'\nEjemplos de review_id repetidos:')
    ids_dupes = df_reviews[df_reviews['review_id'].duplicated(keep=False)].sort_values('review_id')
    display(ids_dupes.head(10))

review_id duplicados: 814

Ejemplos de review_id repetidos:


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequente...",07/03/2018 00:00,20/03/2018 18:08
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequente...",07/03/2018 00:00,20/03/2018 18:08
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,21/09/2017 00:00,26/09/2017 03:27
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,21/09/2017 00:00,26/09/2017 03:27
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixaç...,07/03/2018 00:00,08/03/2018 03:00
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixaç...,07/03/2018 00:00,08/03/2018 03:00
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,02/03/2018 00:00,05/03/2018 01:43
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,02/03/2018 00:00,05/03/2018 01:43
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,"O pedido consta de 2 produtos e até agora recebi apenas 1 produto, e o que m...",09/09/2017 00:00,13/09/2017 09:52
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,"O pedido consta de 2 produtos e até agora recebi apenas 1 produto, e o que m...",09/09/2017 00:00,13/09/2017 09:52


In [62]:
df_reviews['review_creation_date'] = pd.to_datetime(df_reviews['review_creation_date'], errors='coerce')
df_reviews['review_answer_timestamp'] = pd.to_datetime(df_reviews['review_answer_timestamp'], errors='coerce')

print(f'Fechas invalidas en review_creation_date: {df_reviews["review_creation_date"].isna().sum()}')
print(f'Fechas invalidas en review_answer_timestamp: {df_reviews["review_answer_timestamp"].isna().sum()}')
print(f'\nRango de review_creation_date:')
print(f'   Mas antigua: {df_reviews["review_creation_date"].min()}')
print(f'   Mas reciente: {df_reviews["review_creation_date"].max()}')

Fechas invalidas en review_creation_date: 0
Fechas invalidas en review_answer_timestamp: 0

Rango de review_creation_date:
   Mas antigua: 2016-10-02 00:00:00
   Mas reciente: 2018-08-31 00:00:00


In [72]:
todas_medianoche = (df_reviews['review_creation_date'].dt.hour == 0).all()
print(f'Todas las review_creation_date tienen hora 00:00: {todas_medianoche}')

Todas las review_creation_date tienen hora 00:00: False


In [76]:
medianoche_creation = (df_reviews['review_creation_date'].dt.hour == 0).sum()
total = len(df_reviews)

print(f'review_creation_date con hora 00:00: {medianoche_creation:,} ({medianoche_creation/total*100:.1f}%)')

review_creation_date con hora 00:00: 99,139 (99.9%)


In [77]:
no_medianoche = df_reviews[df_reviews['review_creation_date'].dt.hour != 0]
print(f'Reviews con review_creation_date diferente de 00:00: {len(no_medianoche)}')
display(no_medianoche[['review_id', 'order_id', 'review_score', 'review_creation_date', 'review_answer_timestamp']])

Reviews con review_creation_date diferente de 00:00: 85


,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
444,554f6545765e5c000f1751682dd00158,c0f3e5b46a6f8a899bb3a9de4afd5525,5,2017-10-15 01:00:00,2017-10-15 22:24:00
714,9e7431393db15c66d3441e13b69845b4,b599b60a3fa09e7a4003eb264aa99a07,5,2017-10-15 01:00:00,2017-10-16 01:36:00
1498,291f39426317eddf3d70aa3dc9fbdcd6,5798e78e71a22ca63000082377a15073,5,2017-10-15 01:00:00,2017-10-17 11:26:00
4082,e3a081060d6caba024c22dceb1f6398e,83d500e46a95c23dd67eb3cadbd1bd13,5,2017-10-15 01:00:00,2017-10-16 20:59:00
7422,bcf79cd1d5ae6677cec9a946b1c98f10,027fb2504084c231e857f6cf558a9992,5,2017-10-15 01:00:00,2017-10-17 14:00:00
...,...,...,...,...,...
93987,14aa74873105c816c8b166179a1bd0a6,7cbb819f10680054c5b2d2d1992c1826,4,2017-10-15 01:00:00,2017-10-15 17:31:00
94462,9d618c4d881b2b539318f9a7e93ff703,ac1f454dcc22f536ef0bd5fc3d25cc61,4,2017-10-15 01:00:00,2017-10-15 22:52:00
97102,f85ae141a0d912defb1c690f27ce57ce,c147eb82165b5b541fc943d0747eede3,1,2017-10-15 01:00:00,2017-10-16 14:46:00
97687,10ea095607476757f2a8b7f543ac930e,c55e1514675dc10976509a057394e27f,5,2017-10-15 01:00:00,2017-10-17 14:25:00


In [78]:
hora_1 = (df_reviews['review_creation_date'].dt.hour == 1).sum()
print(f'review_creation_date con hora 01:00: {hora_1}')

review_creation_date con hora 01:00: 85


In [79]:
print(f'   Resumen de hallazgos en Reviews:')
print(f'   1. 814 review_id duplicados (mismo review asociado a diferentes ordenes)')
print(f'   2. 58.7% de reviews sin comentario (solo dejaron score)')
print(f'   3. 88.3% sin titulo')
print(f'   4. Fechas como texto — convertir a datetime en Fase 2')
print(f'   5. CSV con saltos de linea en comentarios (filas potencialmente corruptas)')
print(f'   6. 768 ordenes sin review:')
print(f'      - 646 delivered (clientes que no respondieron la encuesta)')
print(f'      - 122 no entregadas (normal que no tengan review)')
print(f'     Rango de fechas coherente (2016-2018)')
print(f'     Sin fechas invalidas')
print(f'      Promedio de score: 4.09 — sesgo positivo (57.8% son score 5)')
print(f'   7. review_creation_date solo registra fecha (99.9% tiene hora 00:00)')
print(f'      Las 85 excepciones son del 2017-10-15 01:00 (anomalia por horario de verano)')
print(f'   → Decision Fase 2: tratar review_creation_date solo como fecha, sin hora')

   Resumen de hallazgos en Reviews:
   1. 814 review_id duplicados (mismo review asociado a diferentes ordenes)
   2. 58.7% de reviews sin comentario (solo dejaron score)
   3. 88.3% sin titulo
   4. Fechas como texto — convertir a datetime en Fase 2
   5. CSV con saltos de linea en comentarios (filas potencialmente corruptas)
   6. 768 ordenes sin review:
      - 646 delivered (clientes que no respondieron la encuesta)
      - 122 no entregadas (normal que no tengan review)
     Rango de fechas coherente (2016-2018)
     Sin fechas invalidas
      Promedio de score: 4.09 — sesgo positivo (57.8% son score 5)
   7. review_creation_date solo registra fecha (99.9% tiene hora 00:00)
      Las 85 excepciones son del 2017-10-15 01:00 (anomalia por horario de verano)
   → Decision Fase 2: tratar review_creation_date solo como fecha, sin hora


### 5.6 Orders (Órdenes) — Tabla Central

In [89]:
perfilar_dataset(df_orders, 'orders')

PERFIL: ORDERS

Dimensiones: 99,441 filas x 8 columnas
Memoria: 57.56 MB

Tipos de datos:
   order_id: str
   customer_id: str
   order_status: str
   order_purchase_timestamp: str
   order_approved_at: str
   order_delivered_carrier_date: str
   order_delivered_customer_date: str
   order_estimated_delivery_date: str

Valores nulos:
   order_approved_at: 160 (0.2%)
   order_delivered_carrier_date: 1,783 (1.8%)
   order_delivered_customer_date: 2,965 (3.0%)

Filas duplicadas completas: 0

Valores unicos por columna:
   order_id: 99,441
   customer_id: 99,441
   order_status: 8
   order_purchase_timestamp: 88,789
   order_approved_at: 50,462
   order_delivered_carrier_date: 61,544
   order_delivered_customer_date: 75,649
   order_estimated_delivery_date: 459

Validacion de IDs:
     order_id: OK (32 chars, hexadecimal)
     customer_id: OK (32 chars, hexadecimal)

Primeras 3 filas:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,02/10/2017 10:56,02/10/2017 11:07,04/10/2017 19:55,10/10/2017 21:25,18/10/2017 00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,24/07/2018 20:41,26/07/2018 03:24,26/07/2018 14:31,07/08/2018 15:27,13/08/2018 00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,08/08/2018 08:38,08/08/2018 08:55,08/08/2018 13:50,17/08/2018 18:06,04/09/2018 00:00


In [90]:
print('Distribucion de order_status:')
total_orders = len(df_orders)
for status, count in df_orders['order_status'].value_counts().items():
    barra = '█' * int(count / total_orders * 50)
    print(f'   {status}: {count:,} ({count/total_orders*100:.1f}%) {barra}')

Distribucion de order_status:
   delivered: 96,478 (97.0%) ████████████████████████████████████████████████
   shipped: 1,107 (1.1%) 
   canceled: 625 (0.6%) 
   unavailable: 609 (0.6%) 
   invoiced: 314 (0.3%) 
   processing: 301 (0.3%) 
   created: 5 (0.0%) 
   approved: 2 (0.0%) 


In [94]:
date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

df_orders_temp = df_orders.copy()
for col in date_cols:
    df_orders_temp[col] = pd.to_datetime(df_orders_temp[col], dayfirst=True, errors='coerce')

print('Rango temporal del dataset:')
print(f'   Primera compra: {df_orders_temp["order_purchase_timestamp"].min()}')
print(f'   Ultima compra: {df_orders_temp["order_purchase_timestamp"].max()}')

print(f'\nNulos en fechas por columna:')
for col in date_cols:
    nulos = df_orders_temp[col].isna().sum()
    print(f'   {col}: {nulos:,} ({nulos/total_orders*100:.1f}%)')

Rango temporal del dataset:
   Primera compra: 2016-09-04 21:15:00
   Ultima compra: 2018-10-17 17:30:00

Nulos en fechas por columna:
   order_purchase_timestamp: 0 (0.0%)
   order_approved_at: 160 (0.2%)
   order_delivered_carrier_date: 1,783 (1.8%)
   order_delivered_customer_date: 2,965 (3.0%)
   order_estimated_delivery_date: 0 (0.0%)


In [98]:
print('Nulos en fechas de entrega por status:')
for col in ['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date']:
    print(f'\n   {col}:')
    nulos_por_status = df_orders_temp[df_orders_temp[col].isna()]['order_status'].value_counts()
    for status, count in nulos_por_status.items():
        print(f'      {status}: {count}')

Nulos en fechas de entrega por status:

   order_approved_at:
      canceled: 141
      delivered: 14
      created: 5

   order_delivered_carrier_date:
      unavailable: 609
      canceled: 550
      invoiced: 314
      processing: 301
      created: 5
      approved: 2
      delivered: 2

   order_delivered_customer_date:
      shipped: 1107
      canceled: 619
      unavailable: 609
      invoiced: 314
      processing: 301
      delivered: 8
      created: 5
      approved: 2


In [97]:
entregas = df_orders_temp.dropna(subset=['order_delivered_customer_date'])
entrega_antes_compra = entregas[
    entregas['order_delivered_customer_date'] < entregas['order_purchase_timestamp']
]
aprobacion_antes_compra = df_orders_temp.dropna(subset=['order_approved_at'])
aprobacion_antes_compra = aprobacion_antes_compra[
    aprobacion_antes_compra['order_approved_at'] < aprobacion_antes_compra['order_purchase_timestamp']
]

print('Consistencia temporal:')
print(f'   Entregas antes de la compra: {len(entrega_antes_compra)}')
print(f'   Aprobaciones antes de la compra: {len(aprobacion_antes_compra)}')

Consistencia temporal:
   Entregas antes de la compra: 0
   Aprobaciones antes de la compra: 0


In [101]:
todas_medianoche = (df_orders_temp['order_estimated_delivery_date'].dt.hour == 0).all()
print(f'Todas las order_estimated_delivery_date tienen hora 00:00: {todas_medianoche}')

Todas las order_estimated_delivery_date tienen hora 00:00: True


In [102]:
print(f'   Resumen de hallazgos en Orders:')
print(f'   1. Fechas en formato DD/MM/YYYY — requiere dayfirst=True al convertir')
print(f'   2. order_estimated_delivery_date solo registra fecha (hora siempre 00:00)')
print(f'   3. 160 ordenes sin fecha de aprobacion (141 canceladas + 14 delivered + 5 created)')
print(f'   4. 8 ordenes delivered sin fecha de entrega al cliente (anomalia menor)')
print(f'   5. 2 ordenes delivered sin fecha de entrega al carrier (anomalia menor)')
print(f'     0 entregas antes de la compra (consistencia temporal OK)')
print(f'     0 aprobaciones antes de la compra')
print(f'      97% de ordenes son delivered')
print(f'      Rango temporal: septiembre 2016 a octubre 2018')

   Resumen de hallazgos en Orders:
   1. Fechas en formato DD/MM/YYYY — requiere dayfirst=True al convertir
   2. order_estimated_delivery_date solo registra fecha (hora siempre 00:00)
   3. 160 ordenes sin fecha de aprobacion (141 canceladas + 14 delivered + 5 created)
   4. 8 ordenes delivered sin fecha de entrega al cliente (anomalia menor)
   5. 2 ordenes delivered sin fecha de entrega al carrier (anomalia menor)
     0 entregas antes de la compra (consistencia temporal OK)
     0 aprobaciones antes de la compra
      97% de ordenes son delivered
      Rango temporal: septiembre 2016 a octubre 2018


### 5.7 Products (Productos)

In [103]:
perfilar_dataset(df_products, 'products')

PERFIL: PRODUCTS

Dimensiones: 32,951 filas x 9 columnas
Memoria: 6.79 MB

Tipos de datos:
   product_id: str
   product_category_name: str
   product_name_lenght: float64
   product_description_lenght: float64
   product_photos_qty: float64
   product_weight_g: float64
   product_length_cm: float64
   product_height_cm: float64
   product_width_cm: float64

Valores nulos:
   product_category_name: 610 (1.9%)
   product_name_lenght: 610 (1.9%)
   product_description_lenght: 610 (1.9%)
   product_photos_qty: 610 (1.9%)
   product_weight_g: 2 (0.0%)
   product_length_cm: 2 (0.0%)
   product_height_cm: 2 (0.0%)
   product_width_cm: 2 (0.0%)

Filas duplicadas completas: 0

Valores unicos por columna:
   product_id: 32,951
   product_category_name: 73
   product_name_lenght: 66
   product_description_lenght: 2,960
   product_photos_qty: 19
   product_weight_g: 2,204
   product_length_cm: 99
   product_height_cm: 102
   product_width_cm: 95

Validacion de IDs:
     product_id: OK (32 chars, 

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,1000.00,30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00



Estadisticas descriptivas (numericas):


,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.00,32341.00,32341.00,32949.00,32949.00,32949.00,32949.00
mean,48.48,771.50,2.19,2276.47,30.82,16.94,23.20
std,10.25,635.12,1.74,4282.04,16.91,13.64,12.08
min,5.00,4.00,1.00,0.00,7.00,2.00,6.00
25%,42.00,339.00,1.00,300.00,18.00,8.00,15.00
50%,51.00,595.00,1.00,700.00,25.00,13.00,20.00
75%,57.00,972.00,3.00,1900.00,38.00,21.00,30.00
max,76.00,3992.00,20.00,40425.00,105.00,105.00,118.00


In [109]:
sin_categoria = df_products['product_category_name'].isna().sum()
print(f'   Productos sin categoría: {sin_categoria}')

print(f'\nTop 10 categorias mas comunes:')
total_productos = len(df_products)
for cat, count in df_products['product_category_name'].value_counts().head(10).items():
    barra = '█' * int(count / total_productos * 50)
    print(f'   {cat}: {count:,} ({count/total_productos*100:.1f}%) {barra}')

   Productos sin categoría: 610

Top 10 categorias mas comunes:
   cama_mesa_banho: 3,029 (9.2%) ████
   esporte_lazer: 2,867 (8.7%) ████
   moveis_decoracao: 2,657 (8.1%) ████
   beleza_saude: 2,444 (7.4%) ███
   utilidades_domesticas: 2,335 (7.1%) ███
   automotivo: 1,900 (5.8%) ██
   informatica_acessorios: 1,639 (5.0%) ██
   brinquedos: 1,411 (4.3%) ██
   relogios_presentes: 1,329 (4.0%) ██
   telefonia: 1,134 (3.4%) █


In [105]:
dim_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
print('Validacion de dimensiones fisicas:')
for col in dim_cols:
    nulos = df_products[col].isna().sum()
    ceros = (df_products[col] == 0).sum()
    print(f'   {col}: nulos={nulos}, ceros={ceros}')

Validacion de dimensiones fisicas:
   product_weight_g: nulos=2, ceros=4
   product_length_cm: nulos=2, ceros=0
   product_height_cm: nulos=2, ceros=0
   product_width_cm: nulos=2, ceros=0


In [106]:
categorias_productos = set(df_products['product_category_name'].dropna().unique())
categorias_traduccion = set(df_category_translation['product_category_name'].unique())

sin_traduccion = categorias_productos - categorias_traduccion
print(f'Categorias en productos sin traduccion al ingles: {len(sin_traduccion)}')
if sin_traduccion:
    for cat in sin_traduccion:
        print(f'   → {cat}')

Categorias en productos sin traduccion al ingles: 2
   → pc_gamer
   → portateis_cozinha_e_preparadores_de_alimentos


In [108]:
print(f'   Resumen de hallazgos en Products:')
print(f'   1. 610 productos sin categoria, nombre, descripcion ni fotos (1.9%)')
print(f'   2. 2 productos sin dimensiones fisicas (peso, largo, alto, ancho)')
print(f'   3. 4 productos con peso = 0g (sospechoso)')
print(f'   4. 2 categorias ausentes en tabla de traduccion:')
print(f'      → pc_gamer (ya es comprensible en ingles)')
print(f'      → portateis_cozinha_e_preparadores_de_alimentos (requiere traduccion)')
print(f'   5. Typo en nombre de columnas: "lenght" en lugar de "length"')
print(f'     product_id unico y valido (32 chars hex)')
print(f'     Sin filas duplicadas')
print(f'   → Decision Fase 2: etiquetar los 610 como "sin_categoria", agregar las 2 traducciones faltantes')

   Resumen de hallazgos en Products:
   1. 610 productos sin categoria, nombre, descripcion ni fotos (1.9%)
   2. 2 productos sin dimensiones fisicas (peso, largo, alto, ancho)
   3. 4 productos con peso = 0g (sospechoso)
   4. 2 categorias ausentes en tabla de traduccion:
      → pc_gamer (ya es comprensible en ingles)
      → portateis_cozinha_e_preparadores_de_alimentos (requiere traduccion)
   5. Typo en nombre de columnas: "lenght" en lugar de "length"
     product_id unico y valido (32 chars hex)
     Sin filas duplicadas
   → Decision Fase 2: etiquetar los 610 como "sin_categoria", agregar las 2 traducciones faltantes


### 5.8 Sellers (Vendedores)

In [110]:
perfilar_dataset(df_sellers, 'sellers')

PERFIL: SELLERS

Dimensiones: 3,095 filas x 4 columnas
Memoria: 0.66 MB

Tipos de datos:
   seller_id: str
   seller_zip_code_prefix: int64
   seller_city: str
   seller_state: str

Sin valores nulos

Filas duplicadas completas: 0

Valores unicos por columna:
   seller_id: 3,095
   seller_zip_code_prefix: 2,246
   seller_city: 611
   seller_state: 23

Validacion de IDs:
     seller_id: OK (32 chars, hexadecimal)

Primeras 3 filas:


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ



Estadisticas descriptivas (numericas):


,seller_zip_code_prefix
count,3095.00
mean,32291.06
std,32713.45
min,1001.00
25%,7093.50
50%,14940.00
75%,64552.50
max,99730.00


In [111]:
print('Top 10 estados por cantidad de vendedores:')
total_sellers = len(df_sellers)
for estado, count in df_sellers['seller_state'].value_counts().head(10).items():
    barra = '█' * int(count / total_sellers * 50)
    print(f'   {estado}: {count:,} ({count/total_sellers*100:.1f}%) {barra}')

Top 10 estados por cantidad de vendedores:
   SP: 1,849 (59.7%) █████████████████████████████
   PR: 349 (11.3%) █████
   MG: 244 (7.9%) ███
   SC: 190 (6.1%) ███
   RJ: 171 (5.5%) ██
   RS: 129 (4.2%) ██
   GO: 40 (1.3%) 
   DF: 30 (1.0%) 
   ES: 23 (0.7%) 
   BA: 19 (0.6%) 


In [113]:
problemas = df_sellers[
    df_sellers['seller_city'].str.contains('√|£|Â|Ã', na=False)
]
print(f'Sellers con problemas de encoding en ciudad: {len(problemas)}')

ciudades = df_sellers['seller_city'].unique()
con_diacriticos = [c for c in ciudades if any(ch in str(c) for ch in 'áéíóúãõâêôçñü')]
print(f'Ciudades con diacriticos: {len(con_diacriticos)}')
if len(con_diacriticos) > 0:
    print(f'Ejemplos:')
    for c in con_diacriticos[:10]:
        print(f'   → {c}')

Sellers con problemas de encoding en ciudad: 0
Ciudades con diacriticos: 0


In [114]:
print(f'   Resumen de hallazgos en Sellers:')
print(f'     Sin valores nulos')
print(f'     Sin filas duplicadas')
print(f'     IDs validos (32 chars hex)')
print(f'     Sin problemas de encoding ni diacriticos en ciudades')
print(f'     Ciudades ya normalizadas en minusculas')
print(f'      59.7% de vendedores concentrados en SP (mas que clientes con 42%)')

   Resumen de hallazgos en Sellers:
     Sin valores nulos
     Sin filas duplicadas
     IDs validos (32 chars hex)
     Sin problemas de encoding ni diacriticos en ciudades
     Ciudades ya normalizadas en minusculas
      59.7% de vendedores concentrados en SP (mas que clientes con 42%)


### 5.9 Category Translation (Traducción de Categorías)

In [116]:
perfilar_dataset(df_category_translation, 'category_translation')

PERFIL: CATEGORY_TRANSLATION

Dimensiones: 71 filas x 2 columnas
Memoria: 0.01 MB

Tipos de datos:
   product_category_name: str
   product_category_name_english: str

Sin valores nulos

Filas duplicadas completas: 0

Valores unicos por columna:
   product_category_name: 71
   product_category_name_english: 71

Primeras 3 filas:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [117]:
typos = {
    'fashio_female_clothing': 'fashion_female_clothing',
    'costruction_tools_garden': 'construction_tools_garden',
    'costruction_tools_tools': 'construction_tools_tools',
    'home_confort': 'home_comfort'
}

print('   Typos detectados en traducciones al ingles:')
for mal, bien in typos.items():
    existe = mal in df_category_translation['product_category_name_english'].values
    print(f'   "{mal}" → deberia ser "{bien}" (encontrado: {existe})')

   Typos detectados en traducciones al ingles:
   "fashio_female_clothing" → deberia ser "fashion_female_clothing" (encontrado: True)
   "costruction_tools_garden" → deberia ser "construction_tools_garden" (encontrado: True)
   "costruction_tools_tools" → deberia ser "construction_tools_tools" (encontrado: True)
   "home_confort" → deberia ser "home_comfort" (encontrado: True)


In [118]:
print(f'   Resumen de hallazgos en Category Translation:')
print(f'   1. 4 typos en traducciones al ingles')
print(f'   2. Faltan 2 categorias que existen en products: pc_gamer, portateis_cozinha_e_preparadores_de_alimentos')
print(f'     Sin nulos ni duplicados')
print(f'     71 categorias traducidas de 73 existentes')
print(f'   → Decision Fase 2: corregir los 4 typos y agregar las 2 traducciones faltantes')

   Resumen de hallazgos en Category Translation:
   1. 4 typos en traducciones al ingles
   2. Faltan 2 categorias que existen en products: pc_gamer, portateis_cozinha_e_preparadores_de_alimentos
     Sin nulos ni duplicados
     71 categorias traducidas de 73 existentes
   → Decision Fase 2: corregir los 4 typos y agregar las 2 traducciones faltantes
